## Setup — Install Libraries & Upload Dataset

In [ ]:
# Install required libraries
# transformers → provides BERT, XLM-RoBERTa, tokenizers, and pipelines
# torch        → deep learning framework used under the hood by transformers
# scikit-learn → provides StratifiedKFold and metrics
!pip install -q transformers torch scikit-learn pandas

In [1]:
# Upload and extract the dataset zip file
from google.colab import files
import zipfile, os

# A file picker dialog will appear — select your zip file
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
print(f"Uploaded: {zip_name}")

# Step 1: extract the outer zip
with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall("/content/dataset")

# Step 2: extract the inner zips (train.csv and test.csv are zipped inside)
inner_folder = "/content/dataset/Basics of BERT and XLM-RoBERTa - PyTorch"

with zipfile.ZipFile(f"{inner_folder}/train.csv.zip", "r") as z:
    z.extractall("/content/dataset")

with zipfile.ZipFile(f"{inner_folder}/test.csv.zip", "r") as z:
    z.extractall("/content/dataset")

print("\nExtracted files:")
print(os.listdir("/content/dataset"))

Saving Basics of BERT and XLM-RoBERTa - PyTorch - 2 (1).zip to Basics of BERT and XLM-RoBERTa - PyTorch - 2 (1) (1).zip
Uploaded: Basics of BERT and XLM-RoBERTa - PyTorch - 2 (1) (1).zip

Extracted files:
['train.csv', 'test.csv', 'Basics of BERT and XLM-RoBERTa - PyTorch']


---
## Task 1 — Understanding BERT and XLM-RoBERTa

Both models are **encoder-only transformers** — they read the entire sentence bidirectionally (left AND right at the same time), making them excellent at *understanding* text.

| | BERT | XLM-RoBERTa |
|---|---|---|
| Created by | Google (2018) | Facebook AI (2019) |
| Languages | English only | 100 languages |
| Vocab size | 30,522 | 250,002 |
| Special tokens | `[CLS]` `[SEP]` `[PAD]` `[MASK]` | `<s>` `</s>` `<pad>` `<mask>` |
| Best for | English NLP tasks | Multilingual tasks |

> **Why XLM-RoBERTa for this challenge?** Our dataset has 15 languages including Arabic, Urdu, Thai, Chinese. BERT would fail on 43% of the test set. XLM-RoBERTa handles all of them natively.

In [3]:
from transformers import BertTokenizer, XLMRobertaTokenizer

# Load both tokenizers from Hugging Face
# 'from_pretrained' downloads the tokenizer vocabulary and config
bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
xlm_tokenizer  = XLMRobertaTokenizer.from_pretrained("xlm-roberta-base")

print("BERT vocab size     :", bert_tokenizer.vocab_size)   # 30,522
print("XLM-RoBERTa vocab  :", xlm_tokenizer.vocab_size)    # 250,002

print("\nBERT special tokens :")
print(bert_tokenizer.special_tokens_map)

print("\nXLM-RoBERTa special tokens:")
print(xlm_tokenizer.special_tokens_map)

BERT vocab size     : 30522
XLM-RoBERTa vocab  : 250002

BERT special tokens :
{'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}

XLM-RoBERTa special tokens:
{'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>'}


---
## Task 2 — Tokenizing Text

- **`input_ids`** → integer ID for each token in the vocabulary
- **`attention_mask`** → 1 for real tokens, 0 for padding
- **`token_type_ids`** → 0 for sentence A tokens, 1 for sentence B tokens (BERT only)

Since this is an NLI task, we always pass **both sentences** together so the model can compare them.

In [5]:
import pandas as pd
from transformers import BertTokenizer

train_df = pd.read_csv("/content/dataset/train.csv")

# Take a real English row from the dataset
english_row = train_df[train_df["lang_abv"] == "en"].iloc[0]
premise    = english_row["premise"]
hypothesis = english_row["hypothesis"]

print("Premise   :", premise)
print("Hypothesis:", hypothesis)
print("Label     :", english_row["label"], "(0=entailment, 1=neutral, 2=contradiction)")

bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")



bert_enc = bert_tokenizer(
    premise,
    hypothesis,
    add_special_tokens=True,      # adds [CLS] and [SEP] automatically
    max_length=128,
    padding="max_length",         # pads to exactly 128 tokens
    truncation=True,              # cuts if combined text > 128 tokens
    return_attention_mask=True,
    return_token_type_ids=True,   # 0=premise, 1=hypothesis
    return_tensors="pt"           # returns PyTorch tensors
)

tokens   = bert_tokenizer.convert_ids_to_tokens(bert_enc["input_ids"][0])
ids      = bert_enc["input_ids"][0].tolist()
mask     = bert_enc["attention_mask"][0].tolist()
type_ids = bert_enc["token_type_ids"][0].tolist()

print("\n=== BERT Tokenization ===")
print(f"{'idx':>4} | {'token':<22} | {'id':>6} | mask | type_id")
print("-" * 55)
for i, (tok, tid, m, t) in enumerate(zip(tokens, ids, mask, type_ids)):
    if tid == 0: break  # stop printing at first [PAD] token
    flag = " ← special" if tok in bert_tokenizer.all_special_tokens else ""
    print(f"{i:>4} | {tok:<22} | {tid:>6} | {m}    | {t}{flag}")

print(f"\nReal tokens: {mask.count(1)}  |  Padding tokens: {mask.count(0)}")

# decode() reverses the process: IDs → readable text
decoded = bert_tokenizer.decode(bert_enc["input_ids"][0], skip_special_tokens=True)
print("\nDecoded back:", decoded[:80], "...")

Premise   : and these comments were considered in formulating the interim rules.
Hypothesis: The rules developed in the interim were put together with these comments in mind.
Label     : 0 (0=entailment, 1=neutral, 2=contradiction)

=== BERT Tokenization ===
 idx | token                  |     id | mask | type_id
-------------------------------------------------------
   0 | [CLS]                  |    101 | 1    | 0 ← special
   1 | and                    |   1998 | 1    | 0
   2 | these                  |   2122 | 1    | 0
   3 | comments               |   7928 | 1    | 0
   4 | were                   |   2020 | 1    | 0
   5 | considered             |   2641 | 1    | 0
   6 | in                     |   1999 | 1    | 0
   7 | formula                |   5675 | 1    | 0
   8 | ##ting                 |   3436 | 1    | 0
   9 | the                    |   1996 | 1    | 0
  10 | interim                |   9455 | 1    | 0
  11 | rules                  |   3513 | 1    | 0
  12 | .           

In [7]:
from transformers import XLMRobertaTokenizer

xlm_tokenizer = XLMRobertaTokenizer.from_pretrained("xlm-roberta-base")

arabic_row = train_df[train_df["lang_abv"] == "ar"].iloc[0]
print("Arabic Premise   :", arabic_row["premise"])
print("Arabic Hypothesis:", arabic_row["hypothesis"])

xlm_enc = xlm_tokenizer(
    arabic_row["premise"],
    arabic_row["hypothesis"],
    add_special_tokens=True,
    max_length=128,
    padding="max_length",
    truncation=True,
    return_attention_mask=True,
    return_tensors="pt"
)

xlm_tokens = xlm_tokenizer.convert_ids_to_tokens(xlm_enc["input_ids"][0])
xlm_ids    = xlm_enc["input_ids"][0].tolist()
xlm_mask   = xlm_enc["attention_mask"][0].tolist()

print("\n=== XLM-RoBERTa Tokenization (Arabic) ===")
print(f"{'idx':>4} | {'token':<25} | {'id':>6} | mask")
print("-" * 48)
for i, (tok, tid, m) in enumerate(zip(xlm_tokens, xlm_ids, xlm_mask)):
    if tid == 1: break  # stop at padding
    flag = " ← special" if tok in xlm_tokenizer.all_special_tokens else ""
    print(f"{i:>4} | {tok:<25} | {tid:>6} | {m}{flag}")

print(f"\nReal tokens: {xlm_mask.count(1)}  |  Padding: {xlm_mask.count(0)}")

decoded_xlm = xlm_tokenizer.decode(xlm_enc["input_ids"][0], skip_special_tokens=True)
print("\nDecoded back:", decoded_xlm[:80], "...")

Arabic Premise   : إذا أمكن ، تعرّف على المؤامرة مسبقًا.
Arabic Hypothesis: حاول أن تفهم الحبكة في البداية، إذا كنت تستطيع.

=== XLM-RoBERTa Tokenization (Arabic) ===
 idx | token                     |     id | mask
------------------------------------------------
   0 | <s>                       |      0 | 1 ← special
   1 | ▁إذا                      |  13231 | 1
   2 | ▁أ                        |   1333 | 1
   3 | مكن                       |  93390 | 1
   4 | ▁،                        |    725 | 1
   5 | ▁تع                       |  21205 | 1
   6 | ر                         |    835 | 1
   7 | ّ                         |   1168 | 1
   8 | ف                         |   2031 | 1
   9 | ▁على                      |    556 | 1
  10 | ▁المؤ                     |  76415 | 1
  11 | امر                       |  76174 | 1
  12 | ة                         |    250 | 1
  13 | ▁م                        |    665 | 1
  14 | سبق                       | 106909 | 1
  15 | ً                         | 

---
## Task 3 — Preparing Input Data for the Model

Key concepts:
- **Padding** (`padding="max_length"`) ensures all sequences in a batch are the same length — required for tensor operations
- **Truncation** (`truncation=True`) cuts sequences that exceed `max_length` so they fit in the model
- **Attention mask** tells the model which tokens are real (1) vs padding (0) — padding tokens are ignored during self-attention
- **`max_length=128`** is safe here because 99%+ of our premise+hypothesis pairs are under 128 tokens

In [9]:
from transformers import XLMRobertaTokenizer
import pandas as pd

train_df      = pd.read_csv("/content/dataset/train.csv")
xlm_tokenizer = XLMRobertaTokenizer.from_pretrained("xlm-roberta-base")

# ── 1. Inspect special tokens and vocab ─────────────────────────
print("Special tokens map:")
for name, token in xlm_tokenizer.special_tokens_map.items():
    print(f"  {name:<20} → {token}")

print(f"\nVocab size: {xlm_tokenizer.vocab_size:,}")

# ── 2. Single sample tokenization with full output ───────────────
row = train_df.iloc[0]

enc = xlm_tokenizer(
    row["premise"],
    row["hypothesis"],
    add_special_tokens=True,
    max_length=128,
    padding="max_length",
    truncation=True,
    return_attention_mask=True,
    return_tensors="pt"
)

print("\nKeys returned       :", list(enc.keys()))
print("input_ids shape     :", enc["input_ids"].shape)        # [1, 128]
print("attention_mask shape:", enc["attention_mask"].shape)   # [1, 128]
print("Real tokens         :", enc["attention_mask"].sum().item())
print("Pad tokens          :", (enc["attention_mask"] == 0).sum().item())

# ── 3. Batch tokenization (efficient — all rows at once) ─────────
# In real training you tokenize the whole dataset, not row by row
def tokenize_batch(premises, hypotheses, tokenizer, max_length=128):
    """Tokenize a batch of premise-hypothesis pairs together."""
    return tokenizer(
        list(premises),
        list(hypotheses),
        add_special_tokens=True,
        max_length=max_length,
        padding="max_length",
        truncation=True,
        return_attention_mask=True,
        return_tensors="pt"
    )

# Test on first 5 rows
batch_enc = tokenize_batch(
    train_df["premise"][:5],
    train_df["hypothesis"][:5],
    xlm_tokenizer
)
print("\nBatch of 5 rows:")
print("  input_ids shape    :", batch_enc["input_ids"].shape)       # [5, 128]
print("  attention_mask shape:", batch_enc["attention_mask"].shape)  # [5, 128]
print("\nAttention masks (1=real token, 0=padding):")
for i, row_mask in enumerate(batch_enc["attention_mask"].tolist()):
    real = sum(row_mask)
    print(f"  Row {i}: {real} real tokens, {128-real} padding tokens")

Special tokens map:
  bos_token            → <s>
  eos_token            → </s>
  unk_token            → <unk>
  sep_token            → </s>
  pad_token            → <pad>
  cls_token            → <s>
  mask_token           → <mask>

Vocab size: 250,002

Keys returned       : ['input_ids', 'attention_mask']
input_ids shape     : torch.Size([1, 128])
attention_mask shape: torch.Size([1, 128])
Real tokens         : 32
Pad tokens          : 96

Batch of 5 rows:
  input_ids shape    : torch.Size([5, 128])
  attention_mask shape: torch.Size([5, 128])

Attention masks (1=real token, 0=padding):
  Row 0: 32 real tokens, 96 padding tokens
  Row 1: 36 real tokens, 92 padding tokens
  Row 2: 37 real tokens, 91 padding tokens
  Row 3: 34 real tokens, 94 padding tokens
  Row 4: 53 real tokens, 75 padding tokens


---
## Task 4 — Loading and Exploring the Dataset

The dataset is **XNLI** — Cross-Lingual Natural Language Inference:
- **Train:** 12,120 rows across 15 languages
- **Test:** 5,195 rows (no labels — you submit predictions)
- **Columns:** `id`, `premise`, `hypothesis`, `lang_abv`, `language`, `label`

In [10]:
import pandas as pd
import matplotlib.pyplot as plt

# ── Load all files ───────────────────────────────────────────────
train_df  = pd.read_csv("/content/dataset/train.csv")
test_df   = pd.read_csv("/content/dataset/test.csv")
sample_df = pd.read_csv("/content/dataset/Basics of BERT and XLM-RoBERTa - PyTorch/sample_submission.csv")

# ── Shape ────────────────────────────────────────────────────────
print("Train shape:", train_df.shape)   # (12120, 6)
print("Test shape :", test_df.shape)    # (5195, 5) — no 'label' column
print("Sample sub :", sample_df.shape)

# ── First rows ───────────────────────────────────────────────────
print("\nFirst 3 rows of train:")
train_df.head(3)

Train shape: (12120, 6)
Test shape : (5195, 5)
Sample sub : (5195, 2)

First 3 rows of train:


,id,premise,hypothesis,lang_abv,language,label
0,5130fd2cb5,and these comments were considered in formulat...,The rules developed in the interim were put to...,en,English,0
1,5b72532a0b,These are issues that we wrestle with in pract...,Practice groups are not permitted to work on t...,en,English,2
2,3931fbe82a,Des petites choses comme celles-là font une di...,J'essayais d'accomplir quelque chose.,fr,French,0


In [11]:
# ── Columns and dtypes ───────────────────────────────────────────
print("Columns:", train_df.columns.tolist())
print("\nDtypes:")
print(train_df.dtypes)

# ── Missing values ───────────────────────────────────────────────
print("\nMissing values per column:")
print(train_df.isnull().sum())
# All zeros → clean dataset, no preprocessing needed for nulls

# ── Label distribution ───────────────────────────────────────────
print("\nLabel distribution:")
label_map = {0: "Entailment", 1: "Neutral", 2: "Contradiction"}
dist = train_df["label"].value_counts().sort_index()
for label, count in dist.items():
    print(f"  {label} ({label_map[label]:<15}): {count:,} ({count/len(train_df)*100:.1f}%)")
# Very balanced! No class weighting needed.

# ── Language distribution ────────────────────────────────────────
print("\nLanguage distribution:")
print(train_df["language"].value_counts().to_string())

# ── Text length analysis (to choose max_length) ──────────────────
train_df["premise_words"]    = train_df["premise"].apply(lambda x: len(str(x).split()))
train_df["hypothesis_words"] = train_df["hypothesis"].apply(lambda x: len(str(x).split()))
train_df["combined_words"]   = train_df["premise_words"] + train_df["hypothesis_words"]

print("\nCombined text length stats (words):")
print(train_df["combined_words"].describe().round(1))
pct_under_128 = (train_df["combined_words"] < 128).mean() * 100
print(f"\n→ {pct_under_128:.1f}% of pairs are under 128 words → max_length=128 is safe")

Columns: ['id', 'premise', 'hypothesis', 'lang_abv', 'language', 'label']

Dtypes:
id            object
premise       object
hypothesis    object
lang_abv      object
language      object
label          int64
dtype: object

Missing values per column:
id            0
premise       0
hypothesis    0
lang_abv      0
language      0
label         0
dtype: int64

Label distribution:
  0 (Entailment     ): 4,176 (34.5%)
  1 (Neutral        ): 3,880 (32.0%)
  2 (Contradiction  ): 4,064 (33.5%)

Language distribution:
language
English       6870
Chinese        411
Arabic         401
French         390
Swahili        385
Urdu           381
Vietnamese     379
Russian        376
Hindi          374
Greek          372
Thai           371
Spanish        366
Turkish        351
German         351
Bulgarian      342

Combined text length stats (words):
count    12120.0
mean        27.2
std         15.5
min          2.0
25%         17.0
50%         26.0
75%         36.0
max        216.0
Name: combined_wo

---
## Task 5 — Creating Cross-Validation Folds

**Why StratifiedKFold?**
- Regular KFold splits randomly and can create unbalanced folds (e.g., fold 1 has 90% entailment)
- `StratifiedKFold` guarantees every fold has the **same label distribution** as the full dataset
- With 5 folds: each fold uses 80% for training (~9,696 rows) and 20% for validation (~2,424 rows)
- `shuffle=True` prevents sorted data from creating biased folds
- `random_state=42` makes the split reproducible every run

In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold

train_df = pd.read_csv("/content/dataset/train.csv")

premises   = train_df["premise"].values
hypotheses = train_df["hypothesis"].values
labels     = train_df["label"].values

# ── Set up StratifiedKFold ───────────────────────────────────────
N_FOLDS = 5
kf = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,       # randomize before splitting
    random_state=42     # seed for reproducibility
)

train_folds = []
val_folds   = []

label_map = {0: "Entailment", 1: "Neutral", 2: "Contradiction"}

# kf.split(X, y) — X can be anything (we use premises), y must be the labels
for fold_num, (train_idx, val_idx) in enumerate(kf.split(premises, labels)):
    train_folds.append(train_idx)
    val_folds.append(val_idx)

    train_labels = labels[train_idx]
    val_labels   = labels[val_idx]

    print(f"\n── Fold {fold_num + 1} ──────────────────────────────────────")
    print(f"   Train: {len(train_idx):,}  |  Val: {len(val_idx):,}")
    for lbl in [0, 1, 2]:
        tr_pct = (train_labels == lbl).mean() * 100
        vl_pct = (val_labels   == lbl).mean() * 100
        print(f"   {label_map[lbl]:<15}: train={tr_pct:.1f}%  val={vl_pct:.1f}%")

print("\n✅ All 5 folds created successfully!")
print("   Each fold shows near-identical label percentages → StratifiedKFold working correctly")


── Fold 1 ──────────────────────────────────────
   Train: 9,696  |  Val: 2,424
   Entailment     : train=34.4%  val=34.5%
   Neutral        : train=32.0%  val=32.0%
   Contradiction  : train=33.5%  val=33.5%

── Fold 2 ──────────────────────────────────────
   Train: 9,696  |  Val: 2,424
   Entailment     : train=34.5%  val=34.4%
   Neutral        : train=32.0%  val=32.0%
   Contradiction  : train=33.5%  val=33.5%

── Fold 3 ──────────────────────────────────────
   Train: 9,696  |  Val: 2,424
   Entailment     : train=34.5%  val=34.4%
   Neutral        : train=32.0%  val=32.0%
   Contradiction  : train=33.5%  val=33.5%

── Fold 4 ──────────────────────────────────────
   Train: 9,696  |  Val: 2,424
   Entailment     : train=34.5%  val=34.4%
   Neutral        : train=32.0%  val=32.0%
   Contradiction  : train=33.5%  val=33.5%

── Fold 5 ──────────────────────────────────────
   Train: 9,696  |  Val: 2,424
   Entailment     : train=34.5%  val=34.4%
   Neutral        : train=32.0%  val

In [13]:
# ── How to USE a specific fold ───────────────────────────────────
# Pick fold 0 (index 0) as an example
fold = 0

X_train_premise    = premises[train_folds[fold]]
X_train_hypothesis = hypotheses[train_folds[fold]]
y_train            = labels[train_folds[fold]]

X_val_premise      = premises[val_folds[fold]]
X_val_hypothesis   = hypotheses[val_folds[fold]]
y_val              = labels[val_folds[fold]]

print(f"Fold {fold} is ready for training:")
print(f"  Training pairs   : {len(X_train_premise):,}")
print(f"  Validation pairs : {len(X_val_premise):,}")
print(f"\nSample training pair:")
print(f"  Premise   : {X_train_premise[0]}")
print(f"  Hypothesis: {X_train_hypothesis[0]}")
print(f"  Label     : {y_train[0]} ({label_map[y_train[0]]})")

Fold 0 is ready for training:
  Training pairs   : 9,696
  Validation pairs : 2,424

Sample training pair:
  Premise   : and these comments were considered in formulating the interim rules.
  Hypothesis: The rules developed in the interim were put together with these comments in mind.
  Label     : 0 (Entailment)


---
## 🚀 Bonus — Full Training Pipeline (Putting It All Together)

This cell shows the complete pipeline: tokenization → dataset → model → training loop using fold 0.

> ⚠️ This requires a GPU and takes ~10-15 minutes per epoch. Make sure `Runtime → Change runtime type → T4 GPU` is selected.

In [ ]:
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import XLMRobertaTokenizer, XLMRobertaForSequenceClassification
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

# ── Config ───────────────────────────────────────────────────────
MODEL_NAME  = "xlm-roberta-base"
MAX_LENGTH  = 128
BATCH_SIZE  = 16
EPOCHS      = 1       # increase to 3-5 for better results
LR          = 2e-5
NUM_LABELS  = 3       # entailment / neutral / contradiction
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# ── Dataset class ─────────────────────────────────────────────────
class NLIDataset(Dataset):
    """Converts premise-hypothesis pairs into tokenized tensors."""
    def __init__(self, premises, hypotheses, labels, tokenizer, max_length):
        self.premises    = premises
        self.hypotheses  = hypotheses
        self.labels      = labels
        self.tokenizer   = tokenizer
        self.max_length  = max_length

    def __len__(self):
        return len(self.premises)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.premises[idx]),
            str(self.hypotheses[idx]),
            add_special_tokens=True,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            return_tensors="pt"
        )
        return {
            "input_ids"      : enc["input_ids"].squeeze(0),
            "attention_mask" : enc["attention_mask"].squeeze(0),
            "label"          : torch.tensor(self.labels[idx], dtype=torch.long)
        }

# ── Load data and tokenizer ───────────────────────────────────────
train_df  = pd.read_csv("/content/dataset/train.csv")
tokenizer = XLMRobertaTokenizer.from_pretrained(MODEL_NAME)

premises   = train_df["premise"].values
hypotheses = train_df["hypothesis"].values
labels     = train_df["label"].values

# ── Create fold 0 ────────────────────────────────────────────────
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds = list(kf.split(premises, labels))
train_idx, val_idx = folds[0]   # use fold 0

train_dataset = NLIDataset(premises[train_idx], hypotheses[train_idx],
                            labels[train_idx], tokenizer, MAX_LENGTH)
val_dataset   = NLIDataset(premises[val_idx],   hypotheses[val_idx],
                            labels[val_idx],   tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}")

# ── Load model ───────────────────────────────────────────────────
model = XLMRobertaForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS
)
model = model.to(DEVICE)

# AdamW is the standard optimizer for transformer fine-tuning
optimizer = AdamW(model.parameters(), lr=LR)

# ── Training loop ─────────────────────────────────────────────────
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for step, batch in enumerate(train_loader):
        input_ids  = batch["input_ids"].to(DEVICE)
        attn_mask  = batch["attention_mask"].to(DEVICE)
        batch_lbls = batch["label"].to(DEVICE)

        optimizer.zero_grad()                         # clear previous gradients
        outputs = model(input_ids=input_ids,
                        attention_mask=attn_mask,
                        labels=batch_lbls)            # loss computed inside model
        loss = outputs.loss
        loss.backward()                               # compute gradients
        optimizer.step()                              # update weights
        total_loss += loss.item()

        if step % 100 == 0:
            print(f"  Epoch {epoch+1} | Step {step}/{len(train_loader)} | Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(train_loader)

    # ── Validation ───────────────────────────────────────────────
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in val_loader:
            input_ids  = batch["input_ids"].to(DEVICE)
            attn_mask  = batch["attention_mask"].to(DEVICE)
            outputs    = model(input_ids=input_ids, attention_mask=attn_mask)
            preds      = torch.argmax(outputs.logits, dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(batch["label"].numpy())

    val_acc = accuracy_score(all_labels, all_preds)
    print(f"\n✅ Epoch {epoch+1} complete | Avg Train Loss: {avg_loss:.4f} | Val Accuracy: {val_acc:.4f}\n")

print("Training complete!")